# Stage-1 PROVENANCE FORENSICS  (resolve before any A/B/C or campaign)

The E6/E7 "zero" verdicts are explained (no E1 reference in the dir + hard collapse
is structurally impossible on 157-step single-corruption streams). But the
previously-reported Stage-1/forward numbers (hard boundaries, slope ~1.11,
`S_frozen=1.185`, R²=0.982) now contradict the probe. **Exactly one of
{reported-real, protocol-mislabel, synthetic-output, probe-wrong} is true.**

Local inspection already settled P1–P3:
- **P1 (synthetic-output): RULED OUT.** Every `--self-test` writes only to a
  `TemporaryDirectory` and prints `[PASS]/[FAIL]` lines — **no report-format
  tables/verdicts** (measured: 22 lines, 0 md-table lines). The full-report
  fake-GPU harnesses live only in a local scratchpad — **not in the repo**
  (`git ls-tree origin/formulation | grep test_|fake` → empty) — and the
  notebook never invokes them. Their synthetic constants (`R_TRUE=12` ⇒ slope
  0.083; `S=0.0875`) do **not** match the reported 1.11 / 1.185 / 0.982.
- **P2 (ephemeral): PLAUSIBLE.** `pstar_colab.ipynb` has `USE_DRIVE=False →
  LOCAL_RESULTS_DIR=/content/pstar_results` (ephemeral). The only location log
  is the manifest, written *inside* the results dir (lost with it). Check (i)
  below reads Drive for surviving analysis artifacts.
- **WRONG-PATH (new): the leading hypothesis now.** The earlier strict search
  walked only `/content/drive/MyDrive` to depth 4 — a **Shared Drive**,
  *Shared-with-me*, or deeper path would have been **missed**. If the user
  supplies the correct folder, the "0 files" reading is simply a wrong
  `RESULTS_DIR`. Set the corrected folder in the config cell's `RESULTS_DIR`,
  then Check (i-b) is DECISIVE via the recorded `n_steps`.
- **P3 (protocol-mislabel): STRUCTURALLY the only reconciler.** The Stage-1
  stream code has a single commit (a08c867), no `cycle/repeat/concat`, slices
  exactly 10,000 imgs ⇒ **max 157 steps at batch 64**. So >157 steps is
  impossible except at smaller batch, and "E1 hard collapse" numbers could
  only have come from the **continual** data that *does* exist (50 runs).

This notebook runs the 4 checks that need Drive + GPU, then P5 verdict +
the Option-C premise test. **Do NOT run E6/E7 or re-freeze until this returns
a verdict.**

## 1. GPU check

In [ ]:
import subprocess
out = subprocess.run(["nvidia-smi"], capture_output=True, text=True)
print(out.stdout if out.returncode == 0 else "WARNING: no GPU (checks iii/iv slow on CPU).")

## 2. Config — `# === EDIT ME ===`

In [ ]:
# === EDIT ME ===========================================================
REPO_URL   = "https://github.com/octadion/heat.git"
REPO_DIR   = "heat"
GIT_BRANCH = ""                      # "" = default branch (must contain Stage-1 scripts)

RESULTS_DIR = "/content/drive/MyDrive/pstar_results"   # the campaign dir under audit
CKPT_DIR    = "/content/drive/MyDrive/heat/experiments/checkpoints"
CKPT_WRN    = CKPT_DIR + "/wrn28_10_final.pt"
C10C_ROOT   = "data/cifar10c"
SEED, SEV, BATCH = 42, 5, 64

S_FROZEN_CLAIM = 1.185               # the number under audit (from the GO instruction)
CONT_SLOPE_CLAIM = 1.11              # reported continual soft slope (WRN)
# =======================================================================
print("RESULTS_DIR =", RESULTS_DIR)

## 3. Mount Drive + 4. Clone repo + 5. CIFAR-10-C + 6. Checkpoint

In [ ]:
import os, subprocess, glob
from google.colab import drive
drive.mount("/content/drive")

if not os.path.isdir(REPO_DIR):
    cmd = ["git", "clone"] + (["--branch", GIT_BRANCH] if GIT_BRANCH else []) + [REPO_URL, REPO_DIR]
    subprocess.run(cmd, check=True)
os.chdir("/content/" + REPO_DIR if not os.path.isabs(REPO_DIR) else REPO_DIR)
REPO_ROOT = os.getcwd()
subprocess.run(["pip", "install", "-q", "-r", "requirements.txt"], check=False)
os.environ["PYTHONPATH"] = REPO_ROOT + os.pathsep + os.environ.get("PYTHONPATH", "")
os.environ["PYTHONUTF8"] = "1"

# CIFAR-10-C (needed for checks iii/iv)
subprocess.run(["python", "scripts/download_cifar10c.py", "--root", C10C_ROOT], check=True)

# Resolve checkpoint (Drive path, or glob-fallback anywhere on Drive)
CKPT = CKPT_WRN if os.path.exists(CKPT_WRN) else next(
    iter(glob.glob("/content/drive/MyDrive/**/wrn28_10_final.pt", recursive=True)), "")
assert CKPT and os.path.exists(CKPT), f"WRN checkpoint not found (looked at {CKPT_WRN})"
print("cwd =", REPO_ROOT, "| checkpoint =", CKPT)

# Checkpoint fingerprint (for P4 provenance: same artifact as the validated sweep?)
import hashlib
h = hashlib.sha256(open(CKPT, "rb").read()).hexdigest()[:16]
print(f"checkpoint sha256[:16] = {h}  size = {os.path.getsize(CKPT)/1e6:.1f} MB")

## Check (i) — P2: what Stage-1/forward artifacts survive on Drive?

If analysis artifacts (`stage1_gbar_law.*`, `stage1_hardslope_frozen.json`,
forward predictions) exist but the run JSONs do **not**, Stage-1 analysis ran
somewhere with data that wasn't persisted here → **real-but-lost**. If nothing
survives, Stage-1/forward never touched this Drive.

In [ ]:
import glob, os
print("--- run JSONs (genuine E1 single-corruption / forward heat runs) ---")
import re
E1 = re.compile(r"^p9_wrn28_10_pstar_(gaussian_noise|elastic_transform|impulse_noise|contrast)_lr[-0-9.eE+]+_p[0-9.]+_seed42_sev5\.json$")
FWD= re.compile(r"^p9_wrn28_10_pstar_(fog|shot_noise)_lr[-0-9.eE+]+_p[0-9.]+_seed42_sev5\.json$")
names = [os.path.basename(f) for f in glob.glob(os.path.join(RESULTS_DIR, "*.json"))]
print("  genuine E1 single-corruption runs:", sum(bool(E1.match(n)) for n in names))
print("  forward (fog/shot_noise) runs    :", sum(bool(FWD.match(n)) for n in names))
print("--- analysis artifacts / manifests (would survive a lost run dir) ---")
for pat in ["stage1_e1_manifest.jsonl","stage1b_manifest.jsonl","stage1b_e*_manifest.jsonl",
            "analysis/stage1_gbar_law.*","analysis/stage1_hardslope_frozen.json",
            "analysis/stage1_softboundary.md","analysis/stage1_forward_verdict.md",
            "analysis/stage1b_predictions.json","analysis/e6_*","analysis/e7_*",
            "analysis/quarantine/*"]:
    for f in sorted(glob.glob(os.path.join(RESULTS_DIR, pat))):
        print("   FOUND:", f)
print("(if the two run-JSON counts are 0 AND no stage1_* analysis artifacts appear,")
print(" real-but-lost is unsupported: Stage-1/forward never wrote here.)")

## Check (i-b) — DECISIVE: recorded `n_steps` of the genuine E1 / forward runs

**Set `RESULTS_DIR` in the config cell to the CORRECTED folder first**, then run
this. It counts genuine E1 single-corruption / forward run JSONs and, for 3
samples of each, dumps the recorded `{n_steps, corruption, eta, p, checkpoint,
out_dir}` plus `first_NaN_step` and hard/soft criterion.

- `n_steps ≈ 157` on genuine E1 runs ⇒ the single-corruption results are **REAL**
  (one severity split = 157 batches). Then the QUESTION FLIPS to the probe: do
  they hard-collapse? (see `first_NaN_step`). If they show `first_NaN ≤ 157`,
  my "157-step can't hard-collapse" reasoning AND the probe must be re-examined
  (checkpoint/seed/code diff) — I flag that as the next investigation.
- `n_steps > 157` ⇒ the runs were longer than a single split (cycling / continual
  fallback / concat) ⇒ **PROTOCOL-MISLABEL stands**.
- The gaussian_noise / eta=2e-3 / p=0 cell is compared directly against my probe
  (n_steps=157, first_NaN=None, acc≈0.73) when present.

In [ ]:
import os, re, glob, json, sys
sys.path.insert(0, REPO_ROOT)
from scripts import pstar_common as pc

RD = RESULTS_DIR   # <-- corrected folder set in the config cell
assert os.path.isdir(RD), f"not a directory: {RD}"
E1  = re.compile(r"^p9_(\w+?)_pstar_(gaussian_noise|elastic_transform|impulse_noise|contrast)_lr([-0-9.eE+]+)_p([0-9.]+)_seed(\d+)_sev(\d+)\.json$")
FWD = re.compile(r"^p9_(\w+?)_pstar_(fog|shot_noise)_lr([-0-9.eE+]+)_p([0-9.]+)_seed(\d+)_sev(\d+)\.json$")
names = sorted(os.path.basename(f) for f in glob.glob(os.path.join(RD, "*.json")))
e1  = [n for n in names if E1.match(n)]
fwd = [n for n in names if FWD.match(n)]
print(f"dir: {RD}")
print(f"genuine E1 single-corruption run JSONs: {len(e1)}")
print(f"forward (fog/shot_noise) run JSONs     : {len(fwd)}")

def report(fn):
    d = json.load(open(os.path.join(RD, fn)))
    a = d.get("args", {}); rows = pc.stream_rows(d)
    summ = pc.heat_summary(d)
    print(f"  {fn}")
    print(f"     n_steps={len(rows)}  first_NaN={pc.first_nonfinite_step(rows)}"
          f"  mean_acc={summ.get('mean_accuracy')}")
    print(f"     corruption={a.get('corruptions')}  eta={a.get('heat_lr')}"
          f"  p={a.get('heat_restore_prob')}  anchor_lambda={a.get('heat_anchor_lambda')}"
          f"  drive={a.get('drive')}")
    print(f"     checkpoint={a.get('checkpoint')}")
    print(f"     out_dir={a.get('out_dir')}")

print("\n=== 3 sample E1 runs (n_steps DECISIVE) ===")
for fn in e1[:3]: report(fn)
print("\n=== 3 sample forward runs ===")
for fn in fwd[:3]: report(fn)

print("\n=== direct comparison to my probe: gaussian_noise eta=2e-3 p=0 ===")
hit = [n for n in e1 if E1.match(n).group(2)=="gaussian_noise"
       and abs(float(E1.match(n).group(3))-2e-3)<1e-9 and float(E1.match(n).group(4))==0.0]
if hit: report(hit[0]); print("   (probe was: n_steps=157, first_NaN=None, acc~0.73)")
else:   print("   no exact gaussian/2e-3/p0 E1 run present to compare.")

nsteps = [len(pc.stream_rows(json.load(open(os.path.join(RD,n))))) for n in e1[:20]]
print(f"\nn_steps across up to 20 E1 runs: min={min(nsteps) if nsteps else 'n/a'}"
      f" max={max(nsteps) if nsteps else 'n/a'}")
print("  ~157 -> single-corruption REAL (flip to probe re-exam).  >157 -> MISLABEL stands.")

## Check (ii) — P5 decisive: hard-only bracket-weighted slope of the 50 CONTINUAL runs

Protocol-mislabel predicts the reported `S_frozen=1.185` / slope 1.11 came from
the continual data. Recompute the bracket-weighted hard-only slope from the
continual runs on disk and compare. A match ⇒ the reported numbers are the
**continual** law, mislabeled as single-corruption E1.

In [ ]:
import sys
sys.path.insert(0, REPO_ROOT)
from scripts import pstar_common as pc
from scripts.analyze_pstar_law import analyze_cell
from scripts.analyze_stage1_followup import wls_line

etas = pc.discover_etas(RESULTS_DIR, "wrn28_10", SEV, SEED)
print("continual etas on disk:", etas)
def collect(hard_only):
    pts = []
    for eta in etas:
        r = analyze_cell(RESULTS_DIR, "wrn28_10", SEV, eta, SEED, hard_only=hard_only)
        lo, hi, ps, x = (r["p_star_bracket_low"], r["p_star_bracket_high"],
                         r["p_star"], r["eta_times_gbar"])
        if ps is not None and x is not None and lo is not None and hi is not None and hi > lo:
            pts.append((x, ps, hi - lo))
    return pts
for label, ho in (("HARD-only", True), ("soft+hard", False)):
    pts = collect(ho)
    if len(pts) < 2:
        print(f"  {label}: only {len(pts)} usable continual point(s) — cannot fit.")
        continue
    w = wls_line([x for x,_,_ in pts], [y for _,y,_ in pts], [1/wd**2 for _,_,wd in pts])
    o = pc.least_squares_line([x for x,_,_ in pts], [y for _,y,_ in pts])
    print(f"  {label}: n={len(pts)} | WLS slope={w['slope']:.4g} ± {w['sigma_slope']:.3g}"
          f" (wR²={w['r2_weighted']:.4f}) | OLS slope={o['slope']:.4g} R²={o['r2']:.4f}")
print(f"\n  COMPARE to claims: S_frozen={S_FROZEN_CLAIM}, continual slope={CONT_SLOPE_CLAIM}, R2=0.982")
print("  If a continual slope ~= a claimed number -> reported 'E1/forward' numbers are the")
print("  continual law (PROTOCOL-MISLABEL confirmed).")

## Check (iii) — P4: symmetry gate (reuses the 50 continual runs; fast)

The known WRN sev-5 anchor must reproduce with this exact checkpoint (p=0 → NaN
~step 603, ‖ḡ‖∈[8,18]). Combined with the probe (single-corruption p=0 does
NOT collapse), this proves both physics coexist ⇒ the probe is correct, not
buggy.

In [ ]:
import subprocess
rc = subprocess.run(["python", "scripts/run_pstar_sweep.py", "--sanity-check-only",
                     "--results-dir", RESULTS_DIR, "--ckpt-wrn", CKPT,
                     "--c10c-root", C10C_ROOT, "--seed", str(SEED),
                     "--batch-size", str(BATCH), "--num-workers", "2"]).returncode
print("\nsanity-gate exit code:", rc, "(0 = continual anchor reproduced with this checkpoint)")

## Check (iv) — Option-C premise: cycled `gaussian_noise ×15`, p=0, η=1e-3

Option C (repeat one corruption so drift accumulates) is only viable if a
cycled single-corruption stream **does** hard-collapse in the ~600-step range,
matching the continual anchor. ~2355 steps (~8–12 min A100). NOTE: the
`per_corruption` summary is distorted by repeating one key, but
`stream_diagnostics` + first-NaN are valid — that is all this probe reads.

In [ ]:
import subprocess, os, json
tag = "probe_cycled15_gn_lr0.001_p0"
outp = os.path.join(RESULTS_DIR, f"p9_wrn28_10_{tag}_seed{SEED}_sev{SEV}.json")
if not os.path.exists(outp):
    subprocess.run(["python", "scripts/run_tier2.py", "--protocol", "p9",
        "--arch", "wrn28_10", "--dataset", "cifar10", "--checkpoint", CKPT,
        "--c10c-root", C10C_ROOT, "--severity", str(SEV), "--seed", str(SEED),
        "--batch-size", str(BATCH), "--corruptions"] + ["gaussian_noise"]*15 +
        ["--methods", "heat", "--heat-lr", "0.001", "--heat-restore-prob", "0",
         "--heat-diagnostic-snapshot", "--variant-tag", tag,
         "--out-dir", RESULTS_DIR], check=True)
from scripts import pstar_common as pc
d = json.load(open(outp))
rows = d["results"]["summary"]["heat"]["stream_diagnostics"]
nan = pc.first_nonfinite_step(rows)
print(f"\ncycled x15: n_steps={len(rows)}  first_NaN_step={nan}")
print("  ~600-range NaN  -> Option C has an empirical basis (drift accumulates, genuine collapse).")
print("  None            -> even cycled single-corruption doesn't collapse -> Option C also fails;")
print("                     the hard boundary may need the true 15-corruption continual (Option B).")

## P5 VERDICT — how to read (i)–(iv)

| Evidence | Reading |
|---|---|
| (i) 0 run JSONs **and** no `stage1_*` analysis artifacts | real-but-lost **unsupported** — Stage-1 never wrote here |
| (i) analysis artifacts present but 0 run JSONs | real-but-lost **plausible** — ran elsewhere, only outputs synced |
| (ii) a continual slope ≈ 1.185 / 1.11 / R²≈0.982 | **PROTOCOL-MISLABEL confirmed** — reported "E1/forward" = the continual law |
| (iii) exit 0 | probe-wrong **ruled out** — both physics coexist under this checkpoint |
| (iv) NaN ~600 | Option **C viable**; None ⇒ C fails, only Option B carries a genuine hard boundary |

Combined with the local P1 (synthetic-output ruled out) and P3 (mislabel is the
only structural reconciler), the expected verdict is **PROTOCOL-MISLABEL**: the
numbers are real *continual* results, and the single-corruption E1/forward
"hard" framing was a labeling error. If so, `S_frozen=1.185` is a **continual**
constant and must not anchor single-corruption predictions.

**Paste the outputs of (i)–(iv). Only after the P5 verdict is fixed do we make
the A/B/C choice — and (iv) decides whether C is even on the table.**